In [ ]:
import random
import chromadb
from sentence_transformers import SentenceTransformer
import ollama

chromadb_client = chromadb.PersistentClient(path="./chroma_db")
collection_name = "local_base"

collection = chromadb_client.get_or_create_collection(name=collection_name)

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

base_templates = [
    (
        "Architect project Opa: mobile app on React Native "
        "(Expo) with navigation, "
        "validation and work with gallery, "
        "React Hook Form"
    ),
    (
        "Backend API service built with ASP.NET Core and Entity Framework, "
        "featuring JWT authentication, role-based access control, "
        "and PostgreSQL database integration"
    ),
    (
        "Multi-container web application deployed via Docker Compose, "
        "using Nginx as a reverse proxy, Flask microservices, "
        "and automated SSL certificate renewal"
    ),
    (
        "Hardware telemetry bridge using two ESP32 microcontrollers "
        "and WT32-ETH01 Ethernet modules to transmit RC control signals "
        "via SBUS and CRSF protocols to a flight controller"
    ),
    (
        "Machine learning pipeline in Python with PyTorch and Scikit-learn "
        "for image classification using ResNet18 transfer learning "
        "and customer clustering analysis"
    )
]

domains = [
    "for e-commerce platform", "for fintech system", "for logistics tracking",
    "for healthcare portal", "for smart home ecosystem", "for educational platform",
    "for cloud analytics", "for autonomous robotics", "for media streaming",
    "for enterprise CRM"
]

extra_features = [
    "with Redis caching and unit tests.",
    "optimized for high-load production traffic.",
    "integrated with CI/CD GitHub Actions pipeline.",
    "featuring real-time WebSocket notifications.",
    "with Prometheus and Grafana monitoring.",
    "supporting multi-language localization.",
    "with strict input sanitization and logging.",
    "deployed on Ubuntu Linux cloud server."
]

random.seed(42)

documents = []

for idx, text in enumerate(base_templates, start=1):
    documents.append((f"doc_{idx}", text))

for i in range(6, 1001):
    base = random.choice(base_templates)
    domain = random.choice(domains)
    feature = random.choice(extra_features)
    documents.append((f"doc_{i}", f"[Module #{i}] {base} {domain} {feature}"))

if collection.count() == 0:
    ids = [item[0] for item in documents]
    texts = [item[1] for item in documents]

    embeddings = embedding_model.encode(texts, batch_size=64, show_progress_bar=True).tolist()

    collection.add(documents=texts, embeddings=embeddings, ids=ids)

user_query = "Which docs with backend?"

query_embendding = embedding_model.encode([user_query]).tolist()

result = collection.query(query_embeddings=query_embendding, n_results=5)

retrieved_chunks = result["documents"][0]

for i, chunk in enumerate(retrieved_chunks):
    print(f"[{i+1}] {chunk}")

context = "\n".join(retrieved_chunks)

promt = f"""Use only the context provided below to answer the question.

If the answer is not in the context, say: "I do not have that information in my base"

Context:
{context}

User query:
{user_query}
"""

responce = ollama.chat(
    model="llama3",
    messages=[
        {
            "role": "system",
            "conte  nt": (
                "You are a clever corporate assistant who answers based on base docs."
            )
        },
        {
            "role": "user",
            "content": promt
        }
    ]
)

print("\n--- Ответ модели ---")
print(responce["message"]["content"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3003.68it/s]


[1] [Module #89] Backend API service built with ASP.NET Core and Entity Framework, featuring JWT authentication, role-based access control, and PostgreSQL database integration for healthcare portal with Prometheus and Grafana monitoring.
[2] [Module #36] Backend API service built with ASP.NET Core and Entity Framework, featuring JWT authentication, role-based access control, and PostgreSQL database integration for healthcare portal with Prometheus and Grafana monitoring.
[3] [Module #998] Backend API service built with ASP.NET Core and Entity Framework, featuring JWT authentication, role-based access control, and PostgreSQL database integration for healthcare portal with Prometheus and Grafana monitoring.
[4] [Module #775] Backend API service built with ASP.NET Core and Entity Framework, featuring JWT authentication, role-based access control, and PostgreSQL database integration for healthcare portal with Prometheus and Grafana monitoring.
[5] [Module #818] Backend API service built wi